In [1]:
pip install yfinance pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
import yfinance as yf
import pandas as pd 

In [3]:
ticker="RELIANCE.NS"
# Fetch 3 years of daily historical data
df = yf.download(ticker, start="2021-01-01", end="2024-01-01")
df.reset_index(inplace=True) #Convert Date from index to standard column
df.to_csv('stock_data.csv',index=False) #Save to physical CSV file without extra row numbers
df=pd.read_csv('stock_data.csv')#Load the saved CSV back into 'df' (Checklist Step 1)
df.head() #preview first 5 rows

[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume
0,NaN,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
1,2021-01-01,897.0470581054688,901.3347986266252,894.5646443140231,897.2726941006515,10015175
2,2021-01-04,898.5590209960938,902.1922975973049,888.2457850636894,900.4772253363753,24513534
3,2021-01-05,887.3882446289062,895.2867576296607,882.8296217394585,888.6971124168427,24123091
4,2021-01-06,863.9860229492188,887.3430480265655,859.8787914584216,887.2979208301023,46401468


In [4]:
# Step 3: Check data types and row/column counts
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 742 entries, 0 to 741
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Date    741 non-null    str  
 1   Close   742 non-null    str  
 2   High    742 non-null    str  
 3   Low     742 non-null    str  
 4   Open    742 non-null    str  
 5   Volume  742 non-null    str  
dtypes: str(6)
memory usage: 98.3 KB


**Fix 1: Drop the Single Empty Date Row**

In [5]:
# 1. Drop rows with missing values
df.dropna(inplace=True)
cols_to_fix=['Close','High','Low','Open','Volume']
for col in cols_to_fix:
    df[col]=pd.to_numeric(df[col], errors='coerce')

In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 741 entries, 1 to 741
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    741 non-null    str    
 1   Close   741 non-null    float64
 2   High    741 non-null    float64
 3   Low     741 non-null    float64
 4   Open    741 non-null    float64
 5   Volume  741 non-null    int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 42.2 KB


In [7]:
df.describe()

,Close,High,Low,Open,Volume
count,741.000000,741.000000,741.000000,741.000000,7.410000e+02
mean,1096.541009,1107.593679,1086.165799,1097.156643,1.510858e+07
std,106.313278,106.464138,105.806177,106.009406,8.859758e+06
min,831.353943,859.811162,825.960347,839.229874,1.705656e+06
25%,1047.791016,1062.545283,1031.478284,1046.240332,9.776989e+06
50%,1116.419922,1128.559837,1106.953353,1117.710725,1.280334e+07
75%,1174.385620,1184.155211,1162.844090,1174.113119,1.765854e+07
max,1291.324341,1297.754090,1278.013144,1293.601566,9.146198e+07


In [8]:
# Step 5: Count missing values in every column
df.isnull().sum()

Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [9]:
# Step 6: Create a categorical 'Trend' column and run value_counts()
df['Trend'] = df.apply(lambda row: 'Up Day' if row['Close'] >= row['Open'] else 'Down Day', axis=1)

# Count the frequency of Up Days vs Down Days
df['Trend'].value_counts()


Trend
Down Day    374
Up Day      367
Name: count, dtype: int64

In [10]:
df.reset_index(drop=True, inplace=True)

In [11]:
# 1. Create empty container
trend_list = []

# 2. Loop through every row position using .iloc
for i in range(len(df)):
    # .iloc[row_position, column_position]
    # In our table: Open is column position 4, Close is column position 1
    close_price = df.iloc[i]['Close']
    open_price = df.iloc[i]['Open']
    
    if close_price >= open_price:
        trend_list.append('Up Day')
    else:
        trend_list.append('Down Day')

# 3. Add as a new column
df['Trend'] = trend_list

# 4. Step 6: Count frequency
df['Trend'].value_counts()

Trend
Down Day    374
Up Day      367
Name: count, dtype: int64

## Phase 1: Data Understanding Report

### 1. Dataset Shape
* **Total Records:** 741 rows (trading days)[cite: 2, 4]
* **Total Features:** 7 columns (Date, Close, High, Low, Open, Volume, Trend)[cite: 2, 4]

### 2. Missing Values Analysis
* **Initial State:** 1 missing record detected in the `Date` column.
* **Resolution:** Dropped the null record using `df.dropna(subset=['Date'])`[cite: 2]. Zero missing values remaining[cite: 2].

### 3. Data Types & Corrections
* **`Date`**: String object (`str`)[cite: 2]
* **Price Fields (`Close`, `High`, `Low`, `Open`)**: Converted from string (`str`) to floating-point numbers (`float64`) using `pd.to_numeric(..., errors='coerce')`[cite: 2].
* **`Volume`**: Converted to 64-bit integer (`int64`)[cite: 2].
* **`Trend`**: Categorical string (`Up Day` / `Down Day`)[cite: 2].

### 4. Key Distributions & Statistics
* **Average Closing Price:** ₹1,096.54 (Standard Deviation: ₹106.31)[cite: 4]
* **Price Range:** Minimum ₹831.35 to Maximum ₹1,291.32[cite: 4]
* **Daily Volume:** Average 15.1 million shares traded (Peak: 91.46 million shares)[cite: 4]
* **Market Trend Breakdown:** 374 Down Days (50.5%) vs. 367 Up Days (49.5%)[cite: 5]

### 5. Identified Data Quality Issues
* Primary issue was incorrect data typing on import (numeric stock prices loaded as string objects)[cite: 2].
* Resolved via explicit type conversion using a loop with `pd.to_numeric()`[cite: 2].

In [12]:
## Phase 2: Data Cleaning & Non-Visual Bivariate Analysis

In [20]:
df.duplicated().sum()

np.int64(0)

In [21]:
df['Trend'].unique()

<ArrowStringArray>
['Down Day', 'Up Day']
Length: 2, dtype: str

In [22]:
df['Date']=pd.to_datetime(df['Date'])

In [19]:
df.dtypes

Date      datetime64[us]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
Trend                str
dtype: object

In [23]:
pd.crosstab(df['Trend'], df['Date'].dt.year)

Date,2021,2022,2023
Trend,,,
Down Day,136,115,123
Up Day,112,133,122


In [28]:
df.groupby('Trend')[['Close', 'Volume']].mean()


,Close,Volume
Trend,,
Down Day,1079.641064,1.506049e+07
Up Day,1113.763297,1.515758e+07


In [29]:
df[['Open', 'High', 'Low', 'Close', 'Volume']].corr()

,Open,High,Low,Close,Volume
Open,1.000000,0.995653,0.995381,0.990279,-0.279259
High,0.995653,1.000000,0.995025,0.996441,-0.252475
Low,0.995381,0.995025,1.000000,0.996175,-0.307683
Close,0.990279,0.996441,0.996175,1.000000,-0.278504
Volume,-0.279259,-0.252475,-0.307683,-0.278504,1.000000


## Phase 2 Summary: Data Cleaning & Bivariate Analysis

### 1. Raw Dataset Audit & Remediation
* **Missing Values:** Audited all features; 1 blank date row identified and removed (`df.dropna()`).
* **Duplicate Check:** Verified 0 duplicate records (`df.duplicated().sum() = 0`).
* **Categorical Standardization:** Verified `Trend` feature categories (`'Down Day'`, `'Up Day'`) with no whitespace or casing errors.
* **Data Types Enforced:** Converted `Date` to `datetime64[ns]` and price string columns to `float64`.

### 2. Non-Visual Bivariate Findings
* **Trend Distribution (Categorical vs Categorical):** 
  * 2021 was slightly bearish (136 Down / 112 Up).
  * 2022 showed strong recovery (115 Down / 133 Up).
  * 2023 was balanced (123 Down / 122 Up).
* **Price & Volume by Trend (Categorical vs Numerical):** Up Days averaged higher closing prices (₹1,113.76 vs ₹1,079.64) and slightly higher trading volumes (15.16M shares vs 15.06M shares).
* **Price-Volume Correlation (Numerical vs Numerical):** Strong positive correlation among price metrics (~0.99), but a weak negative correlation between `Close` price and `Volume` (-0.28), indicating higher trading activity during price dips.

### 3. Readiness Confirmation
The dataset is clean, properly typed, audited for anomalies, and fully ready for visualization and predictive modeling.